In [1]:
import pandas as pd
import sys
from pathlib import Path
import duckdb
import polars as pl
from deltalake import DeltaTable
sys.path.append(str(Path("..").resolve()))
from src.services import bloomberg_client

In [2]:
mypath = r'C:/Users/mmoin/PYTHON PROJECTS/DataWareHouse/bronze/harvest_fund_identifiers'
con = duckdb.connect()
con.execute("INSTALL DELTA;")
con.execute("LOAD DELTA;")
ans = con.execute(f"select ticker from delta_scan('{mypath}') where ticker is not null").pl()
ticker_list = ans.with_columns(pl.col('ticker')+' CN Equity').to_series().to_list()
print(ticker_list)

['TSLY CN Equity', 'TRVL CN Equity', 'TRVL/U CN Equity', 'TRVI CN Equity', 'TEHE CN Equity', 'TDHE CN Equity', 'TBIL CN Equity', 'SUHE CN Equity', 'SOFY CN Equity', 'SHPE CN Equity', 'RYHE CN Equity', 'RDDY CN Equity', 'PRM CN Equity', 'PRM/PR CN Equity', 'PLTE CN Equity', 'ORCY CN Equity', 'NVHE CN Equity', 'NVHE/U CN Equity', 'NVDH/U CN Equity', 'NVDH CN Equity', 'NOVY CN Equity', 'NFLY CN Equity', 'MSTY CN Equity', 'MSTE CN Equity', 'MSHE/U CN Equity', 'MSHE CN Equity', 'MSFH CN Equity', 'MSFH/U CN Equity', 'METE CN Equity', 'LLYH CN Equity', 'LLYH/U CN Equity', 'LLHE/U CN Equity', 'LLHE CN Equity', 'JPHE CN Equity', 'JNJY CN Equity', 'HVOL CN Equity', 'HVOI CN Equity', 'HUTL CN Equity', 'HUTE CN Equity', 'HUBL CN Equity', 'HUBL/U CN Equity', 'HUBL CN Equity', 'HTAE CN Equity', 'HTA CN Equity', 'HTA CN Equity', 'HTA/B CN Equity', 'HTA/U CN Equity', 'HRIF CN Equity', 'HPYT/B CN Equity', 'HPYT CN Equity', 'HPYT CN Equity', 'HPYT/U CN Equity', 'HPYM CN Equity', 'HPYM/U CN Equity', 'HPY

In [4]:
print(ticker_list[0])
# with bloomberg_client.BloombergClient() as client:
#     ref = client.BDS(ticker_list[1],"DVD_HIST_ALL").astype({
#             "Declared Date": "datetime64[ns]",
#             "Ex-Date": "datetime64[ns]",
#             "Record Date": "datetime64[ns]",
#             "Payable Date": "datetime64[ns]",
#             "Dividend Amount": float})
# print(ref.head())

TSLY CN Equity


In [7]:

with bloomberg_client.BloombergClient() as client:
    mydf = pd.DataFrame()
    for fund in ticker_list:
        ref = client.BDS(fund, "DVD_HIST_ALL")
        
        # Only convert columns that exist in the DataFrame
        dtype_map = {
            "Ex-Date": "datetime64[ns]",
            "Record Date": "datetime64[ns]",
            "Payable Date": "datetime64[ns]",
            "Declared Date": "datetime64[ns]",
            "Dividend Amount": float
        }
        
        # Filter to only columns that exist
        dtype_map = {col: dtype for col, dtype in dtype_map.items() if col in ref.columns}
        ref = ref.astype(dtype_map)
        
        mydf = pd.concat([mydf, ref], ignore_index=True)


In [6]:
# First, check the actual columns returned for one fund
with bloomberg_client.BloombergClient() as client:
    test_ref = client.BDS(ticker_list[0], "DVD_HIST_ALL")
    print("Columns in BDS response:")
    print(test_ref.columns.tolist())
    print("\nFirst few rows:")
    print(test_ref.head())


Columns in BDS response:
['Declared Date', 'Ex-Date', 'Record Date', 'Payable Date', 'Dividend Amount', 'Dividend Frequency', 'Dividend Type']

First few rows:
  Declared Date     Ex-Date Record Date Payable Date Dividend Amount  \
0    2026-03-24  2026-03-31  2026-03-31   2026-04-06        0.190000   
1    2026-02-20  2026-02-27  2026-02-27   2026-03-06        0.220000   
2    2026-01-23  2026-01-30  2026-01-30   2026-02-06        0.250000   
3    2025-12-22  2025-12-31  2025-12-31   2026-01-06        0.250000   
4    2025-11-21  2025-11-28  2025-11-28   2025-12-05        0.250000   

  Dividend Frequency Dividend Type  
0            Monthly        Income  
1            Monthly        Income  
2            Monthly        Income  
3            Monthly        Income  
4            Monthly        Income  


In [ ]:
dfa = pd.DataFrame({
    "fund": ["HTA", "HHL", "TRVL"],
    "date": ["2024-01-01", "2024-01-02", "2024-01-03"],
    "NAV": [20, 8, 13]})

dfb = pd.DataFrame({
    "fund": ["HTA", "HTA", "HTA", "HHL", "HHL", "HHL", "TRVL", "TRVL", "TRVL"],
    "date": ["2024-01-01", "2024-01-01", "2024-01-01", "2024-01-02", "2024-01-02", "2024-01-02", "2024-01-03", "2024-01-03", "2024-01-03"],
    "ticker": ["AAPL", "GOOG", "MSFT","GOOG","NVDA","TSLA","AMZN","FB","NFLX"],
    "share_count": [10, 5, 15, 20, 10, 5, 8, 12, 7]})

In [ ]:
df_final = pd.merge(dfb,dfa,on=["fund","date"],how="left")
print(df_final.head(15))

In [ ]:

# Check the consolidated results
print(f"Total rows collected: {len(mydf)}")
print(f"\nColumns in final DataFrame:")
print(mydf.columns.tolist())
print(f"\nData types:")
print(mydf.dtypes)
print(f"\nFirst few rows:")
print(mydf.head(10))
